<a href="https://colab.research.google.com/github/RifaDeen/symAD-ECNN/blob/main/notebooks/models/04_resnet_feature_distance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔬 ResNet Feature Distance Baseline for Brain MRI Anomaly Detection

## 📋 Overview

This notebook implements a **pretrained ResNet-18** baseline for anomaly detection using **feature distance** methods.

### Approach
- **Type**: Transfer Learning + Feature Distance
- **Encoder**: ResNet-18 (pretrained on ImageNet, frozen)
- **Method**: Extract features → KNN/Mahalanobis distance from healthy distribution
- **Training**: NO training required! Just feature extraction.

### Key Features
- **Zero training time** - only feature extraction
- **ImageNet pretrained weights** - transfer learning
- **Multi-layer feature extraction** - captures different abstraction levels
- **KNN and Mahalanobis** - two distance metrics compared

### Expected Performance
- **AUROC**: ~0.75-0.82
- **Training Time**: ~5 min (feature extraction only)

### Research Question
> "Do pretrained ImageNet features alone detect brain anomalies?"

---

## 1️⃣ Setup and Environment Configuration

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

print("✅ Google Drive mounted successfully!")

In [ ]:
# Keep Colab session alive
import IPython
from google.colab import output

display(IPython.display.Javascript('''
 function ClickConnect(){
   btn = document.querySelector("colab-connect-button");
   if (btn != null){
     console.log("Click colab-connect-button");
     btn.click();
   }
   btn = document.querySelector('#ok');
   if (btn != null){
     console.log("Click connect button");
     btn.click();
   }
 }
 setInterval(ClickConnect, 60000)
'''))

print("✅ Keep-alive script activated!")

## ⚡ Turbo Data Loading (Local Disk)

**Why this matters**: Loading 33k+ files from Google Drive is SLOW (~30 min). Instead:
1. **Check** if zip files exist (created once)
2. **Extract** to local runtime disk (~2 min)
3. **Train** with blazing fast I/O (10-20x speedup)

**Note**: If this is your first run, the zip files already exist from CNN-AE training.

In [ ]:
import os
from google.colab import drive

# Check for the zips
base = "/content/drive/MyDrive/symAD-ECNN/data"
zips = [f"{base}/train_fast.zip", f"{base}/val_fast.zip", f"{base}/test_fast.zip"]

missing = [f for f in zips if not os.path.exists(f)]

if len(missing) == 0:
    print("✅ GOOD NEWS: Zip files found! Proceeding to extraction...")
else:
    print("⚠️ WARNING: Zip files missing. Please run the CNN-AE notebook first to create them.")
    print(f"   Missing: {missing}")

In [ ]:
# ==========================================
# ⚡ TURBO LOADER (Unzip to Local)
# ==========================================
import zipfile
import os
import shutil

BASE_DIR = "/content/drive/MyDrive/symAD-ECNN/data"
LOCAL_DIR = "/content/local_data"

ZIPS = {
    "train": f"{BASE_DIR}/train_fast.zip",
    "val":   f"{BASE_DIR}/val_fast.zip",
    "test":  f"{BASE_DIR}/test_fast.zip"
}

print("🚀 Extracting to Local Disk...")

for name, zip_path in ZIPS.items():
    target_dir = f"{LOCAL_DIR}/{name}"
    # Clean up old run if exists
    if os.path.exists(target_dir): 
        shutil.rmtree(target_dir)
    os.makedirs(target_dir, exist_ok=True)

    if os.path.exists(zip_path):
        print(f"   📂 Unzipping {name}...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(target_dir)
    else:
        print(f"   ❌ ERROR: {zip_path} not found!")

print("\n✅ Data Ready! Local folders created.")

In [ ]:
# Import libraries
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve, auc
from sklearn.neighbors import NearestNeighbors
from sklearn.covariance import EmpiricalCovariance
from glob import glob
import os
import time
from tqdm import tqdm
import json

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

print("✅ All libraries imported successfully!")

In [ ]:
# Setup device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Using device: {device}")

if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# Define paths
BASE_PATH = "/content/drive/MyDrive/symAD-ECNN"

# ⚡ DATA PATHS (Point to LOCAL DISK for speed) ⚡
IXI_TRAIN_PATH = "/content/local_data/train"
IXI_VAL_PATH   = "/content/local_data/val"
BRATS_PATH     = "/content/local_data/test"

# Model and results paths (Keep on Drive!)
MODEL_PATH = f"{BASE_PATH}/models/saved_models/resnet_fd"
RESULTS_PATH = f"{BASE_PATH}/results/resnet_feature_distance"

# Create directories
os.makedirs(MODEL_PATH, exist_ok=True)
os.makedirs(RESULTS_PATH, exist_ok=True)

print("📁 Paths configured:")
print(f"   ⚡ Data (Local): {IXI_TRAIN_PATH}")
print(f"   💾 Models (Drive): {MODEL_PATH}")
print(f"   📊 Results (Drive): {RESULTS_PATH}")

In [ ]:
# Data integrity check
print("🔍 Data Integrity Check...")
print("=" * 60)

for name, path in [("IXI Train", IXI_TRAIN_PATH), ("IXI Val", IXI_VAL_PATH), ("BraTS Test", BRATS_PATH)]:
    if os.path.exists(path):
        files = glob(f"{path}/*.npy")
        print(f"✅ {name}: {len(files):,} files")
    else:
        print(f"❌ {name}: Directory NOT FOUND!")

print("-" * 60)
print("✅ Data integrity check complete!")

## 2️⃣ Data Loading

In [ ]:
# Dataset class for ResNet (handles grayscale → RGB conversion)
class MRIDatasetResNet(Dataset):
    """
    Dataset for loading MRI slices and converting to ResNet-compatible format.
    - Converts grayscale (1 channel) to RGB (3 channels)
    - Resizes to 224x224 (ResNet input size)
    - Applies ImageNet normalization
    """
    def __init__(self, file_list):
        self.files = file_list
        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],  # ImageNet means
                std=[0.229, 0.224, 0.225]    # ImageNet stds
            )
        ])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        try:
            # Load grayscale image
            img = np.load(self.files[idx])
            
            # Convert to uint8 for PIL
            img_uint8 = (img * 255).astype(np.uint8)
            
            # Repeat to 3 channels (grayscale → RGB)
            img_rgb = np.stack([img_uint8, img_uint8, img_uint8], axis=-1)
            
            # Apply transforms
            img_tensor = self.transform(img_rgb)
            
            return img_tensor, idx
        except Exception as e:
            print(f"Error loading {self.files[idx]}: {e}")
            return torch.zeros((3, 224, 224)), idx

print("✅ Dataset class defined!")

In [ ]:
# Load file paths
print("📂 Loading file paths...")

train_files = sorted(glob(f"{IXI_TRAIN_PATH}/*.npy"))
val_files = sorted(glob(f"{IXI_VAL_PATH}/*.npy"))
brats_files = sorted(glob(f"{BRATS_PATH}/*.npy"))

# Validation
if len(train_files) == 0:
    raise ValueError(f"❌ No training files found!")

print(f"✅ Found {len(train_files):,} IXI training slices")
print(f"✅ Found {len(val_files):,} IXI validation slices")
print(f"✅ Found {len(brats_files):,} BraTS test slices")

In [ ]:
# Create datasets and dataloaders
BATCH_SIZE = 64

train_dataset = MRIDatasetResNet(train_files)
val_dataset = MRIDatasetResNet(val_files)
test_dataset = MRIDatasetResNet(brats_files)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"✅ DataLoaders created!")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Train batches: {len(train_loader)}")
print(f"   Val batches: {len(val_loader)}")
print(f"   Test batches: {len(test_loader)}")

## 3️⃣ ResNet Feature Extractor

In [ ]:
class ResNetFeatureExtractor(nn.Module):
    """
    ResNet-18 Feature Extractor (pretrained on ImageNet)
    
    Extracts features from multiple layers for richer representation:
    - Layer 2: 128-dim (low-level features)
    - Layer 3: 256-dim (mid-level features)  
    - Layer 4: 512-dim (high-level features)
    
    Total: 896-dim concatenated feature vector
    """
    def __init__(self):
        super(ResNetFeatureExtractor, self).__init__()
        
        # Load pretrained ResNet-18
        resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        
        # Extract layers (freeze all)
        self.conv1 = resnet.conv1
        self.bn1 = resnet.bn1
        self.relu = resnet.relu
        self.maxpool = resnet.maxpool
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4
        
        # Global average pooling
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        
        # Freeze all parameters (no training!)
        for param in self.parameters():
            param.requires_grad = False
    
    def forward(self, x):
        # Initial layers
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        
        # ResNet blocks
        x1 = self.layer1(x)   # 64 channels
        x2 = self.layer2(x1)  # 128 channels
        x3 = self.layer3(x2)  # 256 channels
        x4 = self.layer4(x3)  # 512 channels
        
        # Global average pool each layer
        f2 = self.avgpool(x2).flatten(1)  # 128-dim
        f3 = self.avgpool(x3).flatten(1)  # 256-dim
        f4 = self.avgpool(x4).flatten(1)  # 512-dim
        
        # Concatenate multi-scale features
        features = torch.cat([f2, f3, f4], dim=1)  # 896-dim
        
        return features

# Create model
feature_extractor = ResNetFeatureExtractor().to(device)
feature_extractor.eval()

# Count parameters (all frozen)
total_params = sum(p.numel() for p in feature_extractor.parameters())
trainable_params = sum(p.numel() for p in feature_extractor.parameters() if p.requires_grad)

print("🔬 ResNet-18 Feature Extractor Created!")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,} (all frozen!)")
print(f"   Output feature dimension: 896 (128+256+512)")

## 4️⃣ Feature Extraction

In [ ]:
def extract_features(model, dataloader, device):
    """
    Extract features from all images in dataloader.
    Returns: numpy array of shape (N, 896)
    """
    model.eval()
    all_features = []
    
    with torch.no_grad():
        for data, _ in tqdm(dataloader, desc='Extracting features'):
            data = data.to(device)
            features = model(data)
            all_features.append(features.cpu().numpy())
    
    return np.concatenate(all_features, axis=0)

print("✅ Feature extraction function defined!")

In [ ]:
# Extract features from all datasets
print("🔄 Extracting features from training set (healthy brains)...")
start_time = time.time()
train_features = extract_features(feature_extractor, train_loader, device)
train_time = time.time() - start_time
print(f"   Shape: {train_features.shape}, Time: {train_time:.1f}s")

print("\n🔄 Extracting features from validation set (healthy brains)...")
val_features = extract_features(feature_extractor, val_loader, device)
print(f"   Shape: {val_features.shape}")

print("\n🔄 Extracting features from test set (brain tumors)...")
test_features = extract_features(feature_extractor, test_loader, device)
print(f"   Shape: {test_features.shape}")

print(f"\n✅ Feature extraction complete! Total time: {time.time() - start_time:.1f}s")

## 5️⃣ Anomaly Detection Methods

### 5.1 K-Nearest Neighbors (KNN) Distance

In [ ]:
# Fit KNN on training (healthy) features
print("🔧 Fitting KNN on healthy brain features...")

K = 5  # Number of neighbors
knn = NearestNeighbors(n_neighbors=K, metric='euclidean', n_jobs=-1)
knn.fit(train_features)

print(f"✅ KNN fitted with K={K}")
print(f"   Training samples: {len(train_features):,}")

In [ ]:
# Calculate KNN distances for validation and test sets
print("📏 Computing KNN distances...")

# Validation (normal) - mean distance to K nearest healthy neighbors
val_distances, _ = knn.kneighbors(val_features)
val_knn_scores = val_distances.mean(axis=1)  # Mean of K distances

# Test (anomaly) - mean distance to K nearest healthy neighbors
test_distances, _ = knn.kneighbors(test_features)
test_knn_scores = test_distances.mean(axis=1)

print(f"✅ KNN distances computed!")
print(f"   Normal mean distance: {val_knn_scores.mean():.4f} ± {val_knn_scores.std():.4f}")
print(f"   Anomaly mean distance: {test_knn_scores.mean():.4f} ± {test_knn_scores.std():.4f}")

### 5.2 Mahalanobis Distance

In [ ]:
# Fit Gaussian distribution on training (healthy) features
print("🔧 Fitting Gaussian distribution on healthy brain features...")

# Use empirical covariance with regularization
cov_estimator = EmpiricalCovariance(assume_centered=False)
cov_estimator.fit(train_features)

# Get mean and precision matrix (inverse covariance)
train_mean = train_features.mean(axis=0)
precision = cov_estimator.precision_

print(f"✅ Gaussian fitted!")
print(f"   Feature mean shape: {train_mean.shape}")
print(f"   Precision matrix shape: {precision.shape}")

In [ ]:
def mahalanobis_distance(features, mean, precision):
    """
    Compute Mahalanobis distance from mean.
    D = sqrt((x - μ)' Σ^-1 (x - μ))
    """
    diff = features - mean
    # (N, D) @ (D, D) @ (D, N) -> diagonal gives (N,) distances
    left = np.dot(diff, precision)
    mahal = np.sqrt(np.sum(left * diff, axis=1))
    return mahal

# Calculate Mahalanobis distances
print("📏 Computing Mahalanobis distances...")

val_mahal_scores = mahalanobis_distance(val_features, train_mean, precision)
test_mahal_scores = mahalanobis_distance(test_features, train_mean, precision)

print(f"✅ Mahalanobis distances computed!")
print(f"   Normal mean distance: {val_mahal_scores.mean():.4f} ± {val_mahal_scores.std():.4f}")
print(f"   Anomaly mean distance: {test_mahal_scores.mean():.4f} ± {test_mahal_scores.std():.4f}")

## 6️⃣ Evaluation

In [ ]:
def evaluate_method(normal_scores, anomaly_scores, method_name):
    """
    Evaluate anomaly detection performance.
    Returns: dict with metrics
    """
    # Combine scores and labels
    y_true = np.concatenate([np.zeros(len(normal_scores)), np.ones(len(anomaly_scores))])
    y_scores = np.concatenate([normal_scores, anomaly_scores])
    
    # Calculate metrics
    auroc = roc_auc_score(y_true, y_scores)
    precision, recall, _ = precision_recall_curve(y_true, y_scores)
    auprc = auc(recall, precision)
    
    # Optimal threshold (Youden's J)
    fpr, tpr, thresholds = roc_curve(y_true, y_scores)
    j_scores = tpr - fpr
    optimal_idx = np.argmax(j_scores)
    optimal_threshold = thresholds[optimal_idx]
    
    # Metrics at optimal threshold
    predictions = (y_scores > optimal_threshold).astype(int)
    tp = np.sum((predictions == 1) & (y_true == 1))
    tn = np.sum((predictions == 0) & (y_true == 0))
    fp = np.sum((predictions == 1) & (y_true == 0))
    fn = np.sum((predictions == 0) & (y_true == 1))
    
    accuracy = (tp + tn) / len(y_true)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    f1 = 2 * (prec * rec) / (prec + rec) if (prec + rec) > 0 else 0
    
    results = {
        'method': method_name,
        'auroc': auroc,
        'auprc': auprc,
        'accuracy': accuracy,
        'precision': prec,
        'recall': rec,
        'specificity': spec,
        'f1_score': f1,
        'optimal_threshold': optimal_threshold,
        'confusion_matrix': {'TP': int(tp), 'TN': int(tn), 'FP': int(fp), 'FN': int(fn)}
    }
    
    return results, y_true, y_scores

print("✅ Evaluation function defined!")

In [ ]:
# Evaluate KNN method
print("📊 Evaluating KNN Distance Method...")
knn_results, y_true, knn_y_scores = evaluate_method(val_knn_scores, test_knn_scores, "ResNet-KNN")

print(f"\n{'='*60}")
print(f"📈 KNN DISTANCE RESULTS")
print(f"{'='*60}")
print(f"   AUROC:       {knn_results['auroc']:.4f}")
print(f"   AUPRC:       {knn_results['auprc']:.4f}")
print(f"   Accuracy:    {knn_results['accuracy']:.4f}")
print(f"   Precision:   {knn_results['precision']:.4f}")
print(f"   Recall:      {knn_results['recall']:.4f}")
print(f"   Specificity: {knn_results['specificity']:.4f}")
print(f"   F1-Score:    {knn_results['f1_score']:.4f}")
print(f"{'='*60}")

In [ ]:
# Evaluate Mahalanobis method
print("📊 Evaluating Mahalanobis Distance Method...")
mahal_results, _, mahal_y_scores = evaluate_method(val_mahal_scores, test_mahal_scores, "ResNet-Mahalanobis")

print(f"\n{'='*60}")
print(f"📈 MAHALANOBIS DISTANCE RESULTS")
print(f"{'='*60}")
print(f"   AUROC:       {mahal_results['auroc']:.4f}")
print(f"   AUPRC:       {mahal_results['auprc']:.4f}")
print(f"   Accuracy:    {mahal_results['accuracy']:.4f}")
print(f"   Precision:   {mahal_results['precision']:.4f}")
print(f"   Recall:      {mahal_results['recall']:.4f}")
print(f"   Specificity: {mahal_results['specificity']:.4f}")
print(f"   F1-Score:    {mahal_results['f1_score']:.4f}")
print(f"{'='*60}")

In [ ]:
# Compare both methods
print("\n" + "="*70)
print("📊 METHOD COMPARISON")
print("="*70)
print(f"{'Metric':<15} {'KNN':<15} {'Mahalanobis':<15} {'Better'}")
print("-"*70)

metrics = ['auroc', 'auprc', 'accuracy', 'precision', 'recall', 'specificity', 'f1_score']
for metric in metrics:
    knn_val = knn_results[metric]
    mahal_val = mahal_results[metric]
    better = "KNN ✓" if knn_val > mahal_val else "Mahal ✓" if mahal_val > knn_val else "Tie"
    print(f"{metric:<15} {knn_val:<15.4f} {mahal_val:<15.4f} {better}")

print("="*70)

# Select best method
if knn_results['auroc'] >= mahal_results['auroc']:
    best_method = "KNN"
    best_results = knn_results
    best_scores = knn_y_scores
else:
    best_method = "Mahalanobis"
    best_results = mahal_results
    best_scores = mahal_y_scores

print(f"\n🏆 Best Method: {best_method} (AUROC = {best_results['auroc']:.4f})")

## 7️⃣ Visualization

In [ ]:
# Plot ROC curves for both methods
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curves
fpr_knn, tpr_knn, _ = roc_curve(y_true, knn_y_scores)
fpr_mahal, tpr_mahal, _ = roc_curve(y_true, mahal_y_scores)

axes[0].plot(fpr_knn, tpr_knn, linewidth=2, label=f'KNN (AUROC={knn_results["auroc"]:.4f})')
axes[0].plot(fpr_mahal, tpr_mahal, linewidth=2, label=f'Mahalanobis (AUROC={mahal_results["auroc"]:.4f})')
axes[0].plot([0, 1], [0, 1], 'k--', label='Random')
axes[0].set_xlabel('False Positive Rate', fontsize=12)
axes[0].set_ylabel('True Positive Rate', fontsize=12)
axes[0].set_title('ROC Curves: ResNet Feature Distance', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Score distributions (best method)
if best_method == "KNN":
    normal_scores = val_knn_scores
    anomaly_scores = test_knn_scores
else:
    normal_scores = val_mahal_scores
    anomaly_scores = test_mahal_scores

axes[1].hist(normal_scores, bins=50, alpha=0.7, label='Normal', density=True, color='blue')
axes[1].hist(anomaly_scores, bins=50, alpha=0.7, label='Anomaly', density=True, color='red')
axes[1].axvline(best_results['optimal_threshold'], color='green', linestyle='--', 
                linewidth=2, label=f'Threshold ({best_results["optimal_threshold"]:.2f})')
axes[1].set_xlabel(f'{best_method} Distance', fontsize=12)
axes[1].set_ylabel('Density', fontsize=12)
axes[1].set_title(f'Score Distribution ({best_method})', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_PATH}/resnet_fd_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✅ Evaluation plots saved!")

In [ ]:
# Confusion Matrix for best method
from sklearn.metrics import confusion_matrix
import seaborn as sns

predictions = (best_scores > best_results['optimal_threshold']).astype(int)
cm = confusion_matrix(y_true, predictions)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Normal', 'Anomaly'], yticklabels=['Normal', 'Anomaly'],
            annot_kws={'size': 14, 'weight': 'bold'})
axes[0].set_xlabel('Predicted', fontsize=12, fontweight='bold')
axes[0].set_ylabel('True', fontsize=12, fontweight='bold')
axes[0].set_title(f'Confusion Matrix ({best_method})', fontsize=13, fontweight='bold')

# Normalized
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues', ax=axes[1],
            xticklabels=['Normal', 'Anomaly'], yticklabels=['Normal', 'Anomaly'],
            annot_kws={'size': 14, 'weight': 'bold'})
axes[1].set_xlabel('Predicted', fontsize=12, fontweight='bold')
axes[1].set_ylabel('True', fontsize=12, fontweight='bold')
axes[1].set_title('Confusion Matrix (Normalized)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{RESULTS_PATH}/resnet_fd_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 8️⃣ t-SNE Visualization

In [ ]:
from sklearn.manifold import TSNE

print("🔄 Running t-SNE on ResNet features...")
print("   (This may take 1-2 minutes)")

# Sample for efficiency
max_samples = 500
normal_sample = val_features[:max_samples]
anomaly_sample = test_features[:max_samples]

all_features = np.vstack([normal_sample, anomaly_sample])
labels = np.array([0] * len(normal_sample) + [1] * len(anomaly_sample))

# Run t-SNE
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
features_2d = tsne.fit_transform(all_features)

print("✅ t-SNE complete!")

# Plot
plt.figure(figsize=(10, 8))
scatter = plt.scatter(features_2d[:, 0], features_2d[:, 1], c=labels, cmap='coolwarm', alpha=0.6, s=30)
plt.colorbar(scatter, label='0=Normal, 1=Anomaly')
plt.xlabel('t-SNE Dimension 1', fontsize=12)
plt.ylabel('t-SNE Dimension 2', fontsize=12)
plt.title('ResNet Feature Space (t-SNE)', fontsize=14, fontweight='bold')

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='blue', markersize=10, label='Normal'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='red', markersize=10, label='Anomaly')
]
plt.legend(handles=legend_elements, loc='best', fontsize=11)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_PATH}/resnet_fd_tsne.png', dpi=150, bbox_inches='tight')
plt.show()

## 9️⃣ Save Results

In [ ]:
# Save all results
total_time = time.time() - start_time

final_results = {
    'model': 'ResNet-18 Feature Distance',
    'best_method': best_method,
    'knn_results': knn_results,
    'mahalanobis_results': mahal_results,
    'best_auroc': float(best_results['auroc']),
    'best_auprc': float(best_results['auprc']),
    'feature_dim': 896,
    'total_time_seconds': total_time,
    'training_required': False,
    'pretrained_on': 'ImageNet'
}

with open(f'{RESULTS_PATH}/resnet_fd_results.json', 'w') as f:
    json.dump(final_results, f, indent=4, default=float)

# Save features for potential future use
np.save(f'{RESULTS_PATH}/train_features.npy', train_features)
np.save(f'{RESULTS_PATH}/val_features.npy', val_features)
np.save(f'{RESULTS_PATH}/test_features.npy', test_features)

print("💾 Results saved!")
print(f"   Results JSON: {RESULTS_PATH}/resnet_fd_results.json")
print(f"   Features saved for future use")

In [ ]:
# Final Summary
print("\n" + "="*70)
print("🎉 RESNET FEATURE DISTANCE BASELINE COMPLETE!")
print("="*70)
print(f"\n📊 Best Method: {best_method}")
print(f"   AUROC:       {best_results['auroc']:.4f}")
print(f"   AUPRC:       {best_results['auprc']:.4f}")
print(f"   Accuracy:    {best_results['accuracy']:.4f}")
print(f"   Recall:      {best_results['recall']:.4f}")
print(f"   Specificity: {best_results['specificity']:.4f}")
print(f"\n⏱️ Total Time: {total_time:.1f} seconds ({total_time/60:.1f} minutes)")
print(f"🚫 Training Required: NO (pretrained features only!)")
print("\n" + "="*70)
print("📝 KEY INSIGHT:")
print("   ImageNet-pretrained features can detect brain anomalies")
print("   WITHOUT any medical image training!")
print("="*70)